# Samsung Innovation Campus (SIC) - Big Data Capstone
## Phân Loại & Phát Hiện Nhật Ký Bất Thường Bằng PySpark MLlib (Random Forest)

Notebook này thực hiện huấn luyện mô hình phân loại **RandomForestClassifier** trên nền tảng **Apache Spark MLlib** để phát hiện các request bất thường từ nhật ký web server.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import IndexToString, StringIndexer, VectorIndexer, VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession\
    .builder\
    .appName("RandomForestClassifier")\
    .getOrCreate()

In [ ]:
data = spark.read.options(header=True, inferSchema=True)\
    .csv("web_log_anomaly_dataset.csv")
data.show(5)

In [ ]:
data = data.withColumn("response_time_ms", data["response_time_ms"].cast(DoubleType())).dropna()
data = data.drop('log_id')
data.show(5)

In [ ]:
# danh sach cot
label_col = "is_anomaly"
categorical_cols = [f.name for f in data.schema.fields if f.dataType.typeName() == 'string' and f.name != label_col]
numeric_cols = [f.name for f in data.schema.fields if f.dataType.typeName() in ['integer', 'double']]

# fit label
labelIndexer = StringIndexer(inputCol=label_col, outputCol="indexedLabel").fit(data)
data_new = labelIndexer.transform(data)

In [ ]:
indexed_cat_cols = [c + "Index" for c in categorical_cols]
features = StringIndexer(inputCols=categorical_cols, outputCols=indexed_cat_cols).fit(data_new)
data_new = features.transform(data_new)

assemblerInput = indexed_cat_cols + numeric_cols
assembler = VectorAssembler(inputCols=assemblerInput, outputCol="features")
data_new = assembler.transform(data_new)

data_new.show(5)

In [ ]:
(trainingData, testData) = data_new.randomSplit([0.7, 0.3], seed=42)

In [ ]:
# bat dau train
rf = RandomForestClassifier(labelCol="indexedLabel", featuresCol="features", numTrees=100)
rfModel = rf.fit(trainingData)
# du doan
predictions = rfModel.transform(testData)

# transform
labelConverter = IndexToString(inputCol="prediction", outputCol="predictedLabel", labels=labelIndexer.labels)
predictions = labelConverter.transform(predictions)
# sua lai ten cot true label
predictions.select("predictedLabel", label_col, "features").show(5)

evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Do chinh xac = {accuracy:.2f}")

In [ ]:
# Lay danh sach cac cot trong mohinh
feature_names = assemblerInput

# Muc do quan trong cua tung cot
importances = rfModel.featureImportances

print("\nMuc do anh huong den viec phat hien bat thuong")
for i in range(len(feature_names)):
    print(f"Cột {feature_names[i]:<25}: {importances[i]:.4f}")